# LLM Classification su dataset PWC Multiclass (LM Studio)
Questo notebook implementa un classificatore basato su un LLM in esecuzione locale tramite LM Studio (modello "mlx-community/Ministral 3 3B Instruct 2512"). L'obiettivo è ri-classificare o testare la robustezza di una sottosezione del dataset `pwc_ai_multiclass_balanced.csv`.

In [7]:
import pandas as pd
import json
import time
import os
import signal
from openai import OpenAI
from tqdm.notebook import tqdm

# Inizializzazione del client OpenAI per puntare a LM Studio
# LM Studio, di default, espone le API compatibili con OpenAI su http://localhost:1234/v1
client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

In [8]:
# Test di connessione per verificare quale modello è caricato
try:
    models = client.models.list()
    print("Modelli disponibili:")
    for m in models.data:
        print(f"- {m.id}")
    SUPPORTED_MODEL = models.data[0].id if models.data else "ministral-3-3b-instruct-2512"
except Exception as e:
    print(f"Errore di connessione a LM Studio: {e}")
    SUPPORTED_MODEL = "ministral-3-3b-instruct-2512"

Modelli disponibili:
- ministral-3-3b-instruct-2512
- qwen/qwen3-vl-8b
- text-embedding-nomic-embed-text-v1.5


In [9]:
# Caricamento del dataset bilanciato completo (100k record, 10 classi x 10k)
df_full = pd.read_csv('../data/processed/pwc_ai_multiclass_balanced.csv')
df_full['original_idx'] = df_full.index

print(f"Dataset completo: {df_full.shape}")
print(f"\nDistribuzione classi:")
print(df_full['label'].value_counts())
display(df_full.head(3))

Dataset completo: (100000, 4)

Distribuzione classi:
label
Automotive & UVs         10000
Data Science             10000
Enterprise               10000
Environment              10000
Fintech                  10000
Healthcare               10000
Media & Entertainment    10000
Research                 10000
Robotics & Industry      10000
Virtual Assistants       10000
Name: count, dtype: int64


,description,short_abstract,label,original_idx
0,we build on the theory of ontology logs (ologs...,NaN,Automotive & UVs,0
1,the ability to detect and track the dynamic ob...,"different from existing gmot methods, which tr...",Automotive & UVs,1
2,strong demand for autonomous vehicles and the ...,NaN,Automotive & UVs,2


## Prompt Engineering per il modello Ministral 3B
Il prompt utilizza:
1. **Descrizioni per categoria** con keyword disambiguanti (derivate dalle stesse regex usate in notebook 08 per creare le label)
2. **10 few-shot examples** (uno per categoria) per ancorare il modello 3B
3. **Regola di disambiguazione esplicita**: focalizzarsi sul dominio applicativo, non sulla metodologia ML

Nota: il testo viene troncato a 800 caratteri — i primi 800 char di un abstract contengono i segnali chiave del dominio e riducono la confusione per un modello 3B.

In [10]:
ALLOWED_CATEGORIES = [
    "Automotive & UVs", "Data Science", "Enterprise", "Environment", 
    "Fintech", "Healthcare", "Media & Entertainment", "Research", 
    "Robotics & Industry", "Virtual Assistants"
]

# Few-shot examples estratti dal dataset reale (uno per categoria)
FEW_SHOT_EXAMPLES = [
    ("wogan is an online test generation algorithm based on wasserstein GANs for testing the AI of a self-driving car in the SBST 2022 CPS tool competition.", "Automotive & UVs"),
    ("this short paper describes our ongoing research on greenhouse - a zero-positive machine learning system for time-series anomaly detection.", "Data Science"),
    ("this document represents the ADAPT centre's submission regarding the public consultation on implementation of the EU AI Act for enterprise and trade.", "Enterprise"),
    ("artificial intelligence and machine learning are increasingly seen as key technologies for building more decentralised and resilient energy grids.", "Environment"),
    ("stock market prediction is the act of trying to determine the future value of a company stock or other financial instrument traded on a financial exchange.", "Fintech"),
    ("this report assesses different machine learning approaches to 10-year survival prediction of breast cancer patients.", "Healthcare"),
    ("clone a voice in 5 seconds to generate arbitrary speech in real-time for media applications.", "Media & Entertainment"),
    ("we propose a proportional integral derivative (PID) controller-based coaching scheme to expedite reinforcement learning (RL).", "Research"),
    ("thesis document of the degree of master of science in robotics of Carnegie Mellon University school of computer science.", "Robotics & Industry"),
    ("we propose a chatbot, namely MoChA, to make good use of relevant entities when generating responses in conversational AI.", "Virtual Assistants"),
]

def create_prompt(description):
    system_prompt = """You are a text classifier. Classify the following scientific paper abstract into EXACTLY ONE category.

CATEGORIES AND THEIR DOMAIN SIGNALS:

- Automotive & UVs: autonomous driving, self-driving, vehicles, drones, UAVs, traffic, ADAS, navigation, transportation, automotive
- Data Science: data mining, anomaly detection, feature engineering, AutoML, big data, databases, analytics, time series, clustering, predictive modeling, data warehouse
- Enterprise: business processes, RPA, marketing, HR, recruitment, CRM, customer management, e-commerce, sales, advertising, workflow
- Environment: climate, agriculture, sustainability, energy grids, weather, pollution, farming, ecology, renewable energy, solar, carbon, water, forestry
- Fintech: finance, banking, trading, stocks, fraud detection, credit, cryptocurrency, insurance, investment, portfolio, payments, market risk
- Healthcare: medical, clinical, disease, patient, EHR, genomics, cancer, diagnosis, therapy, surgery, pathology, health monitoring, drug discovery
- Media & Entertainment: video, music, movies, games/gaming, deepfake, fake news, multimedia, streaming, content creation, animation, audio, speech synthesis, voice cloning
- Research: PURE ML/AI methodology with NO specific application domain — new architectures, loss functions, optimization algorithms, theoretical proofs, benchmark papers (CIFAR, ImageNet, MNIST), general NLP/CV methodology
- Robotics & Industry: robots, manufacturing, supply chain, IoT, predictive maintenance, warehouse, factory, production, automation, logistics, control systems, machinery
- Virtual Assistants: chatbot, dialogue systems, conversational AI, speech recognition, voice assistants, LLMs, question answering, NLP applications, language models, text-to-speech, dialog state tracking

CRITICAL DISAMBIGUATION RULE:
- A paper that APPLIES machine learning to solve a problem in a specific domain (healthcare, finance, environment, etc.) belongs to THAT DOMAIN, not Research.
- A paper that proposes new ML methodology tested only on generic benchmarks (CIFAR, ImageNet, MNIST, GLUE) = Research.
- Focus on the APPLICATION DOMAIN, not the methodology used.

Reply with ONLY the exact category name. Nothing else."""

    # Costruzione few-shot examples
    examples_text = "\n".join(
        f'{i+1}. "{ex[0]}" -> {ex[1]}' 
        for i, ex in enumerate(FEW_SHOT_EXAMPLES)
    )
    
    user_prompt = f"""Examples:
{examples_text}

Now classify this abstract:
"{description}"

Category:"""
    
    return system_prompt, user_prompt

In [11]:
# Funzione di classificazione robusta
def classify_record(description, max_retries=3):
    system_p, user_p = create_prompt(description)
    
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=SUPPORTED_MODEL,
                messages=[
                    {"role": "system", "content": system_p},
                    {"role": "user", "content": user_p}
                ],
                temperature=0.0,
                max_tokens=15
            )
            
            result = response.choices[0].message.content.strip().strip('\'"').replace('\n', '')
            
            # 1. Match esatto (case-insensitive)
            result_lower = result.lower().strip()
            for cat in ALLOWED_CATEGORIES:
                if cat.lower() == result_lower:
                    return cat
            
            # 2. Substring match — preferisci il match più lungo
            #    (evita che "Research" matchi quando "Robotics & Industry" è presente)
            matches = [cat for cat in ALLOWED_CATEGORIES if cat.lower() in result_lower]
            if matches:
                return max(matches, key=len)
                    
            return f"UNKNOWN_{result}"
            
        except Exception as e:
            print(f"Errore al tentativo {attempt + 1}: {e}")
            time.sleep(2)
            
    return "ERROR_MAX_RETRIES"

In [12]:
# === Inference Loop con salvataggio incrementale ===
out_path = '../data/processed/llm_predictions_multiclass.csv'

# Se esiste già un file parziale, riprendiamo da dove eravamo rimasti
if os.path.exists(out_path):
    df_done = pd.read_csv(out_path)
    done_ids = set(df_done['original_idx'].tolist())
    print(f"Trovati {len(done_ids)} record già classificati, riprendo da dove ero rimasto...")
else:
    df_done = pd.DataFrame()
    done_ids = set()
    # Scrivi header
    pd.DataFrame(columns=['description', 'short_abstract', 'label', 'original_idx', 'llm_label']).to_csv(out_path, index=False)

# Filtra i record ancora da classificare
df_todo = df_full[~df_full['original_idx'].isin(done_ids)]
print(f"Record da classificare: {len(df_todo)} / {len(df_full)}")

# Classifica con salvataggio ogni 100 record (resiliente a crash)
BATCH_SAVE = 100
buffer = []

for i, row in tqdm(df_todo.iterrows(), total=len(df_todo), desc="Classificazione"):
    desc = str(row['short_abstract'])[:800]  # Tronca a 800 char come da nota
    llm_label = classify_record(desc)
    
    buffer.append({
        'description': row['description'],
        'short_abstract': row['short_abstract'],
        'label': row['label'],
        'original_idx': row['original_idx'],
        'llm_label': llm_label
    })
    
    if len(buffer) >= BATCH_SAVE:
        pd.DataFrame(buffer).to_csv(out_path, mode='a', header=False, index=False)
        buffer = []

# Salva gli ultimi record rimasti nel buffer
if buffer:
    pd.DataFrame(buffer).to_csv(out_path, mode='a', header=False, index=False)

print(f"\nClassificazione completata! Risultati salvati in: {out_path}")

Trovati 0 record già classificati, riprendo da dove ero rimasto...
Record da classificare: 100000 / 100000


Classificazione:   0%|          | 0/100000 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Ricarichiamo dal file salvato per la valutazione finale
df_results = pd.read_csv(out_path)
df_results = df_results.drop_duplicates(subset='original_idx', keep='last')

matched = (df_results['label'] == df_results['llm_label']).sum()
total = len(df_results)
print(f"Accuracy complessiva: {matched}/{total} ({matched/total*100:.2f}%)")

# Verifica record sconosciuti / malformattati
unknown_preds = df_results[df_results['llm_label'].str.startswith('UNKNOWN_', na=False)]
error_preds = df_results[df_results['llm_label'] == 'ERROR_MAX_RETRIES']
print(f"Errori di formattazione output: {len(unknown_preds)}")
print(f"Errori di connessione (max retries): {len(error_preds)}")

# Accuracy per classe
print("\nAccuracy per classe:")
for cat in ALLOWED_CATEGORIES:
    mask = df_results['label'] == cat
    if mask.sum() > 0:
        cat_acc = (df_results.loc[mask, 'label'] == df_results.loc[mask, 'llm_label']).mean()
        print(f"  {cat:<25} {cat_acc*100:.1f}% (n={mask.sum()})")

# Classification report e confusion matrix
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

valid_mask = df_results['llm_label'].isin(ALLOWED_CATEGORIES)
df_valid = df_results[valid_mask]

print(f"\nClassification Report ({len(df_valid)} record validi su {total}):")
print(classification_report(df_valid['label'], df_valid['llm_label'], 
                            labels=ALLOWED_CATEGORIES, zero_division=0))

# Confusion Matrix
cm = confusion_matrix(df_valid['label'], df_valid['llm_label'], labels=ALLOWED_CATEGORIES)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=ALLOWED_CATEGORIES, yticklabels=ALLOWED_CATEGORIES)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - LLM Classification')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

NameError: name 'out_path' is not defined